# Modul B · Kapitel 1.4 — Self-Consistency

## Challenge: Mehrere Reasoning-Pfade per Mehrheit zusammenführen


**Lernziel:** Du entwickelst Self-Consistency auf Basis eines Few-Shot-CoT-Prompts und vergleichst die Mehrheit mehrerer Läufe mit einer einzelnen Antwort auf 20 Testfällen.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

Es gibt genau **eine Challenge**: Self-Consistency implementieren.


---
## 0 · Setup

▶️ Das Setup lädt dieselben verbesserten Few-Shot-CoT-Bausteine wie im vorherigen Notebook und ein größeres Set mit 20 unbekannten Login-Zählaufgaben.

`hole_antwort()` liest die Zahl hinter `ANSWER:`. `mehrheit()` bestimmt aus mehreren Zahlen den häufigsten Wert. Beide Funktionen sind vorbereitet, damit sich die Challenge nur auf Self-Consistency konzentriert.

`MAX_ANTWORT_TOKENS = 2400` begrenzt nur die erzeugte Antwort, nicht den Prompt. Das Limit verhindert endlose oder unnötig teure Ausgaben. Es ist hier viermal so groß wie der Helfer-Default, damit jeder gesampelte CoT-Pfad noch bis zur Zeile `ANSWER:` kommen kann.


In [ ]:
# ▶️ Pakete, Pfade und Modellzugang
import re
import sys
from collections import Counter
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                  Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

from helfer import BASIS_URL, MODELL, frage_llm, lade_daten

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")


In [ ]:
# ▶️ Testdaten und vorbereitete Auswertung
TESTFAELLE = lade_daten("03_login_testfaelle")
ERLAUBTE_ANTWORTEN = [str(n) for n in range(21)]
MAX_ANTWORT_TOKENS = 2400  # Vierfaches des Helfer-Defaults.


def formatiere_aufgabe(fall):
    return f"Question: {fall['frage']}\n\nLog:\n{fall['log']}"


def hole_antwort(text):
    treffer = re.findall(r"ANSWER\s*:\s*(\d+)", text, flags=re.IGNORECASE)
    return treffer[-1] if treffer else None


def mehrheit(antworten):
    stimmen = Counter(a for a in antworten if a is not None)
    return stimmen.most_common(1)[0][0] if stimmen else None


assert len(TESTFAELLE) == 20
assert hole_antwort("Reasoning\nANSWER: 3") == "3"
assert mehrheit(["2", "3", "2"]) == "2"
print(f"{len(TESTFAELLE)} Testfälle geladen.")


In [ ]:
# ▶️ Drei vorgegebene Few-Shot-CoT-Beispiele
BEISPIELE = [
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts came from 10.0.0.5?",
            "log": (
                "09:00 status=failed account=root src=10.0.0.5 method=password\n"
                "09:01 status=failed account=root src=10.0.0.9 method=password\n"
                "09:02 status=success account=root src=10.0.0.5 method=password\n"
                "09:03 status=failed account=admin src=10.0.0.5 method=publickey"
            ),
        }),
        "reasoning": (
            "Required: status=failed and src=10.0.0.5. "
            "Line 1 matches, count 1. Line 2 has the wrong source, count 1. "
            "Line 3 has status=success, count 1. Line 4 matches, count 2."
        ),
        "antwort": "2",
    },
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts for admin used publickey?",
            "log": (
                "10:00 status=failed account=admin src=10.0.0.1 method=publickey\n"
                "10:01 status=failed account=root src=10.0.0.2 method=publickey\n"
                "10:02 status=failed account=admin src=10.0.0.3 method=password\n"
                "10:03 status=success account=admin src=10.0.0.4 method=publickey"
            ),
        }),
        "reasoning": (
            "Required: status=failed, account=admin, and method=publickey. "
            "Line 1 matches, count 1. Line 2 has the wrong account, count 1. "
            "Line 3 has the wrong method, count 1. Line 4 has status=success, count 1."
        ),
        "antwort": "1",
    },
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts for root came from 192.0.2.4 after 18:00?",
            "log": (
                "17:59 status=failed account=root src=192.0.2.4 method=password\n"
                "18:01 status=failed account=root src=192.0.2.4 method=password\n"
                "18:02 status=failed account=admin src=192.0.2.4 method=password\n"
                "18:03 status=failed account=root src=192.0.2.5 method=password"
            ),
        }),
        "reasoning": (
            "Required: after 18:00, status=failed, account=root, and src=192.0.2.4. "
            "Line 1 is too early, count 0. Line 2 matches, count 1. "
            "Line 3 has the wrong account, count 1. Line 4 has the wrong source, count 1."
        ),
        "antwort": "1",
    },
]

print(BEISPIELE[0]["aufgabe"])
print("\n# Reasoning\n" + BEISPIELE[0]["reasoning"])
print("ANSWER:", BEISPIELE[0]["antwort"])


In [ ]:
# ▶️ Der Few-Shot-CoT-Prompt als Ausgangspunkt
def baue_few_shot_cot(beispiele, aufgabe):
    """Baut einen Few-Shot-CoT-Prompt aus Beispielen und einer neuen Aufgabe."""
    anweisung = (
        "Solve the new task by following the exact filtering pattern in the examples. "
        "A field is a filter only when the question explicitly names it; ignore all "
        "other fields. Values such as admin and administrator are different. List every "
        "required condition first, check each line exactly once, keep a running count, "
        "and always finish with ANSWER: <integer>. Be concise."
    )
    bloecke = [
        f"# Task\n{b['aufgabe']}\n\n# Reasoning\n{b['reasoning']}\nANSWER: {b['antwort']}"
        for b in beispiele
    ]
    neue_aufgabe = f"# Task\n{aufgabe}\n\n# Reasoning\n"
    return anweisung + "\n\n" + "\n\n".join(bloecke + [neue_aufgabe])


---
## 1 · Ausgangspunkt: eine Few-Shot-CoT-Antwort

📖 Der Few-Shot-CoT-Prompt wird zunächst einmal mit `temperature=0.0` ausgeführt. Das ergibt pro Testfall genau einen Reasoning-Pfad und eine Endantwort.

Ein einzelner Pfad kann eine Bedingung übersehen oder falsch zählen. Self-Consistency stellt dieselbe Frage mehrfach mit höherer `temperature`. Anschließend entscheidet die häufigste Endantwort.


In [ ]:
# ▶️ Ein einzelner Lauf
BEISPIELFALL = TESTFAELLE[0]
prompt = baue_few_shot_cot(BEISPIELE, formatiere_aufgabe(BEISPIELFALL))
einzeltext = frage_llm(prompt, temperature=0.0, max_tokens=MAX_ANTWORT_TOKENS)

print(f"Fall: {BEISPIELFALL['id']}")
print(f"Soll: {BEISPIELFALL['antwort']}")
print(f"Vorhersage: {hole_antwort(einzeltext)}")
print("\n" + einzeltext)


---
## 2 · Self-Consistency entwickeln

📖 Self-Consistency besteht aus drei Schritten:

1. denselben Few-Shot-CoT-Prompt `k`-mal mit erhöhter `temperature` ausführen,
2. aus jedem Reasoning-Pfad die Endantwort lesen,
3. per Mehrheit eine gemeinsame Vorhersage bestimmen.

Bei `k=3` entstehen beispielsweise die Stimmen `['2', '3', '2']`. Das Self-Consistency-Ergebnis ist dann `2`.

### 🛠️ Challenge 1: `self_consistency()` implementieren

Implementiere `self_consistency(aufgabe, k=3, temperature=0.8)`.

Die Funktion soll:

- den Few-Shot-CoT-Prompt für `aufgabe` bauen,
- das Modell `k`-mal aufrufen,
- mit `hole_antwort()` alle Endantworten sammeln,
- mit `mehrheit()` den Gewinner bestimmen,
- ein Dictionary mit `gewinner`, `stimmen` und `antworttexte` zurückgeben.


In [ ]:
def self_consistency(aufgabe, k=3, temperature=0.8):
    """Sampelt mehrere Reasoning-Pfade und entscheidet per Mehrheit."""
    # TODO: Prompt bauen, k Antworten sammeln und die Mehrheit bestimmen.
    raise NotImplementedError("Challenge 1: Self-Consistency implementieren")


In [ ]:
# ✅ Selbsttest — führt drei Modellaufrufe aus
probe = self_consistency(TESTFAELLE[0], k=3)
assert set(probe) == {"gewinner", "stimmen", "antworttexte"}
assert len(probe["stimmen"]) == 3, "Bei k=3 werden drei Stimmen erwartet."
assert len(probe["antworttexte"]) == 3, "Zu jeder Stimme gehört ein Antworttext."
assert probe["gewinner"] in ERLAUBTE_ANTWORTEN + [None]
print("✅ Challenge 1 gelöst")
print("Stimmen:", probe["stimmen"])
print("Mehrheit:", probe["gewinner"])
print("Sollwert:", TESTFAELLE[0]["antwort"])


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def self_consistency(aufgabe, k=3, temperature=0.8):
    prompt = baue_few_shot_cot(BEISPIELE, formatiere_aufgabe(aufgabe))
    antworttexte = [
        frage_llm(prompt, temperature=temperature, max_tokens=MAX_ANTWORT_TOKENS)
        for _ in range(k)
    ]
    stimmen = [hole_antwort(text) for text in antworttexte]
    return {
        "gewinner": mehrheit(stimmen),
        "stimmen": stimmen,
        "antworttexte": antworttexte,
    }
```

</details>


---
## 3 · Vergleich auf 20 Testfällen

📖 Für jeden Testfall vergleichen wir:

- **Few-Shot-CoT:** ein Lauf bei `temperature=0.0`,
- **Self-Consistency:** drei Läufe bei `temperature=0.8`, danach Mehrheitsentscheid.

Nach jedem vollständig berechneten Testfall erscheint ein Update wie `7/20 computed`. So bleibt bei lokalen Modellen sichtbar, wie weit der Lauf ist.


In [ ]:
# ▶️ Messlauf mit laufenden Updates
K = 3
VERGLEICH = []

for nummer, fall in enumerate(TESTFAELLE, start=1):
    prompt = baue_few_shot_cot(BEISPIELE, formatiere_aufgabe(fall))
    einzeltext = frage_llm(prompt, temperature=0.0, max_tokens=MAX_ANTWORT_TOKENS)
    einzeln = hole_antwort(einzeltext)
    sc = self_consistency(fall, k=K)
    VERGLEICH.append({
        "id": fall["id"],
        "soll": fall["antwort"],
        "few_shot": einzeln,
        "self_consistency": sc["gewinner"],
        "stimmen": sc["stimmen"],
    })
    print(f"{nummer}/{len(TESTFAELLE)} computed", flush=True)

print("Messlauf abgeschlossen.")


In [ ]:
# ▶️ Ergebnisse als Tabelle
treffer_few = sum(r["few_shot"] == r["soll"] for r in VERGLEICH)
treffer_sc = sum(r["self_consistency"] == r["soll"] for r in VERGLEICH)

print(f"{'Fall':<6} {'Soll':>5} {'Few-Shot':>10} {'Self-Cons.':>12}  Stimmen")
print("─" * 62)
for r in VERGLEICH:
    print(f"{r['id']:<6} {r['soll']:>5} {str(r['few_shot']):>10} "
          f"{str(r['self_consistency']):>12}  {r['stimmen']}")

print("─" * 62)
print(f"Few-Shot-CoT:     {treffer_few:>2}/{len(VERGLEICH)} = {treffer_few / len(VERGLEICH):.0%}")
print(f"Self-Consistency: {treffer_sc:>2}/{len(VERGLEICH)} = {treffer_sc / len(VERGLEICH):.0%}")


📖 **Auswertung:** Self-Consistency hilft nur, wenn unterschiedliche Pfade unterschiedliche Fehler machen und die richtige Antwort häufiger vorkommt. Sind die Pfade systematisch falsch, bestätigt die Mehrheit den Fehler. Deshalb zeigt die Tabelle neben dem Gewinner immer auch die drei einzelnen Stimmen.

Die 20 Fälle sind umfangreicher als der erste Vergleich, bleiben aber ein Lehrdatensatz. Die gemessenen Quoten beschreiben diesen Modelllauf und sind keine allgemeine Garantie.


---
## 4 · Was du gebaut hast

- Der verbesserte Few-Shot-CoT-Prompt ist die Einzelantwort-Baseline.
- `self_consistency()` sampelt drei Reasoning-Pfade und bildet die Mehrheit.
- Der Messlauf vergleicht beide Verfahren auf 20 identischen Testfällen.
- Fortschrittsmeldungen zeigen während der Berechnung den aktuellen Stand.

Self-Consistency ist kein zusätzlicher Prompt-Trick. Es ist ein Auswertungsverfahren über mehrere vollständige CoT-Läufe.
